# Hierarchical Clustering

## Learning Objectives

1. **Describe** agglomerative (bottom-up) hierarchical clustering
2. **Implement** single-link, complete-link, and average-link variants
3. **Build** and interpret a dendrogram
4. **Analyze** the $O(n^2 \log n)$ time complexity and scale limitations

## Algorithm: Agglomerative (Bottom-Up)

1. Start: each point is its own cluster ($n$ clusters)
2. Repeat until 1 cluster remains:
   a. Find the two **closest** clusters
   b. Merge them

**What is "closest"?** Depends on the linkage criterion:

| Linkage | $d(A, B)$ | Shape |
|---------|----------|-------|
| **Single** | $\min_{a \in A, b \in B} d(a,b)$ | Long chains |
| **Complete** | $\max_{a \in A, b \in B} d(a,b)$ | Compact, similar-radius |
| **Average** (UPGMA) | $\frac{1}{|A||B|}\sum_{a,b} d(a,b)$ | Balanced |
| **Ward** | Increase in within-cluster variance | Spherical, uniform |

In [1]:
import math

def dist(p, q):
    return math.sqrt(sum((a-b)**2 for a,b in zip(p,q)))

def cluster_dist(A_pts, B_pts, linkage='complete'):
    dists = [dist(a,b) for a in A_pts for b in B_pts]
    if linkage == 'single':   return min(dists)
    if linkage == 'complete': return max(dists)
    if linkage == 'average':  return sum(dists)/len(dists)
    raise ValueError(linkage)

def hierarchical(points, linkage='complete'):
    n = len(points)
    clusters = {i: [points[i]] for i in range(n)}
    history = []
    while len(clusters) > 1:
        ids = sorted(clusters.keys())
        best = (float('inf'), None, None)
        for i in range(len(ids)):
            for j in range(i+1, len(ids)):
                d = cluster_dist(clusters[ids[i]], clusters[ids[j]], linkage)
                if d < best[0]: best = (d, ids[i], ids[j])
        d, a, b = best
        new_id = max(clusters.keys()) + 1
        clusters[new_id] = clusters[a] + clusters[b]
        history.append((a, b, d, len(clusters[new_id])))
        del clusters[a]; del clusters[b]
    return history

import random
rng = random.Random(5)
# 3 clusters of 5 points each
points = []
for cx, cy in [(0,0),(8,0),(4,7)]:
    for _ in range(5):
        points.append([cx+rng.gauss(0,1), cy+rng.gauss(0,1)])

# Labels for display
print("Complete-link dendrogram (merge order):")
print(f"""{'Step':>4} {'Merged':>12} {'Distance':>10} {'Size':>6}""")
history = hierarchical(points, linkage='complete')
for step, (a, b, d, sz) in enumerate(history, 1):
    print(f"{step:>4} ({a:>2},{b:>2})       {d:>10.3f} {sz:>6}")

Complete-link dendrogram (merge order):
Step       Merged   Distance   Size
   1 (10,11)            0.120      2
   2 ( 5, 8)            0.455      2
   3 (12,15)            0.571      3
   4 ( 6, 7)            0.721      2
   5 ( 3, 4)            0.752      2
   6 ( 1, 2)            0.814      2
   7 (14,17)            1.266      4
   8 (16,18)            1.997      4
   9 ( 0,20)            2.175      3
  10 (13,21)            2.439      5
  11 ( 9,22)            2.681      5
  12 (19,23)            2.756      5
  13 (24,25)            9.011     10
  14 (26,27)           11.142     15


In [2]:
# Compare linkages
print("Merges for single vs complete linkage:")
print("""
Single-link:""")
h_single = hierarchical(points, 'single')
for (a,b,d,sz) in h_single[:6]:
    print(f"  merge ({a},{b}) at d={d:.3f}")

print("""
Complete-link:""")
h_complete = hierarchical(points, 'complete')
for (a,b,d,sz) in h_complete[:6]:
    print(f"  merge ({a},{b}) at d={d:.3f}")

# Cut at k=3 clusters
def cut_dendrogram(history, k, n_original):
    "Return which original merge step produces k clusters."
    # Total merges = n-1; merge step i reduces cluster count by 1
    # We want n-k merges
    n_merges = n_original - k
    return history[:n_merges]

merges_for_3 = cut_dendrogram(h_complete, 3, len(points))
print(f"""
Merges to get k=3 clusters: {len(merges_for_3)}""")
print("(These merges produce 3 cluster groups)")

Merges for single vs complete linkage:

Single-link:
  merge (10,11) at d=0.120
  merge (5,8) at d=0.455
  merge (12,15) at d=0.502
  merge (6,7) at d=0.721
  merge (3,4) at d=0.752
  merge (1,2) at d=0.814

Complete-link:
  merge (10,11) at d=0.120
  merge (5,8) at d=0.455
  merge (12,15) at d=0.571
  merge (6,7) at d=0.721
  merge (3,4) at d=0.752
  merge (1,2) at d=0.814

Merges to get k=3 clusters: 12
(These merges produce 3 cluster groups)


## Complexity and Scale

| Step | Cost |
|------|------|
| Initial pairwise distances | $O(n^2)$ |
| Naive find-min each step | $O(n^2)$ per step × $n$ steps = $O(n^3)$ |
| With priority queue (Prim-like) | $O(n^2 \log n)$ |
| Space | $O(n^2)$ for distance matrix |

For $n = 10^5$: $O(n^2) = 10^{10}$ — too large.

**CURE algorithm** (covered next) addresses this: randomly sample points and use representatives to approximate hierarchical clustering for large datasets.